# DEEP BATCH ACTIVE LEARNING BY DIVERSE, UNCERTAIN GRADIENT LOWER BOUNDS

**Jordan T. Ash, Chicheng Zhang, Akshay Krishnamurthy, John Langford, and Alekh Agarwal (2020), International Conference on Learning Representations.**

BADGE represents every unlabeled example with a predicted-label, final-layer gradient embedding, then uses seeded k-MEANS++ to acquire a batch that is both uncertain and diverse.

### What this notebook gives you
- A single-seed, end-to-end BADGE acquisition tutorial on MNIST.
- Direct demonstrations of gradient embeddings and k-MEANS++ seeding.
- A reusable `select_batch` call for a trained multiclass classifier and an unlabeled tensor.

### Two ways to use this
- Run the notebook as-is to see one BADGE learning curve.
- Import `select_batch` and the package helpers in your own active-learning loop.

### What this notebook does **not** do
- It is a tutorial, not a benchmark reproduction or a comparison of methods.
- It uses one seed, while the paper repeats experiments five times.
- It uses five acquisition rounds rather than the paper-scale 349-round condition.
- It uses a 12,000-example smoke pool rather than each paper benchmark's full training pool.
- It uses the bundled two-layer MLP rather than reproducing all paper architectures and benchmark conditions.
- It does not reproduce the paper's 231 experiments, multiple datasets, architectures, or batch-size conditions.

## Contents

- [0. Install dependencies (first run only)](#sec-0-install-dependencies-first-run-only)
- [1. Setup](#sec-1-setup)
- [2. Parameters](#sec-2-parameters)
  - [Optional: scale up to paper-faithful values](#sec-optional-scale-up-to-paper-faithful-values)
- [3. The setup pieces](#sec-3-the-setup-pieces)
  - [3.1 Data](#sec-31-data)
  - [3.2 Model](#sec-32-model)
  - [3.3 Training](#sec-33-training)
  - [3.4 Bootstrap labeled set](#sec-34-bootstrap-labeled-set)
- [4. The BADGE: Batch Active Learning by Diverse Gradient Embeddings method ⭐](#sec-4-the-badge-batch-active-learning-by-diverse-gradient-embeddings-method)
  - [4.1 Intuition](#sec-41-intuition)
  - [4.2 Hallucinated gradient embeddings ⭐](#sec-42-hallucinated-gradient-embeddings)
  - [4.3 k-MEANS++ gradient-batch seeding](#sec-43-k-means-gradient-batch-seeding)
  - [4.4 Putting it together](#sec-44-putting-it-together)
- [5. Running active learning end-to-end](#sec-5-running-active-learning-end-to-end)
  - [5.1 The acquisition loop](#sec-51-the-acquisition-loop)
  - [5.2 Learning curve](#sec-52-learning-curve)
- [6. Use your own data](#sec-6-use-your-own-data)

<a id="sec-0-install-dependencies-first-run-only"></a>

## 0. Install dependencies (first run only)

Run this once in a fresh environment. `%pip` installs into this notebook's kernel.

In [1]:
%pip install -r requirements.txt

…[771 chars stripped for review]…



…[863 chars stripped for review]…
.14/site-packages (from jupyter-server<3,>=2.4.0->jupyterlab->jupyter>=1.0->-r requirements.txt (line 4)) (2.1.0)


…[765 chars stripped for review]…
upyterlab->jupyter>=1.0->-r requirements.txt (line 4)) (1.5.1)



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


<a id="sec-1-setup"></a>

## 1. Setup

In [2]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.nn import functional as F

from method import (
    GradientEmbeddingMLP,
    build_model,
    compute_gradient_embeddings,
    kmeans_plus_plus_seeding,
    load_data,
    select_batch,
    train_from_scratch,
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

<a id="sec-2-parameters"></a>

## 2. Parameters

The table records provenance: `paper` values are paper-stated, `system_default` values make this laptop-sized tutorial practical, and `system_inferred` values fill a paper-unspecified implementation choice.

| Parameter | Variable from paper | Value from paper | Paper value | System value | Where in paper | Used? | Notes |
|---|:-:|:-:|---|---|---|:-:|---|
| `batch_size` | ✅ | ✅ | 100 | — | Algorithm 1 and Section 4 | ✅ | batch size B varying from  $\{100,1000,10000\}$ . |
| `num_rounds` | ✅ | ✅ | 349 | 5 | — | ✅ | Paper runs 349 rounds (~34,900 labels at batch_size=100). At smoke scale we... |
| `initial_labeled` | ✅ | ✅ | 100 | — | Algorithm 1 and Section 4 | ✅ | M=100 being the number of initial random labeled examples |
| `pool_size` | ✅ | ✅ | 'full training set' | 12000 | — | ✅ | Paper uses the full training set (typically 60k–73k samples) as the... |
| `learning_rate` | ✅ | ✅ | 0.001 | — | Section 4 | ✅ | Adam optimizer at 0.001. Spec training.learning_rate: '0.001 for image data... |
| `max_epochs` | ❌ | ❌ | — | 50 | — | ✅ | AUTO-RAISED (A001): max_epochs raised from 8 to 50. The training-budget... |
| `train_until_accuracy` | ✅ | ✅ | 0.99 | — | Section 4 EXPERIMENTS | ✅ | Paper-stated training threshold: 'models using cross-entropy loss and the... |
| `hidden_dim` | ❌ | ❌ | — | 256 | — | ✅ | Paper states no data-type-keyed MLP width. We use hidden_dim=256 as the... |

In [3]:
params = {
    "batch_size": {
        "value": 100,
        "source": 'paper',
        "paper_section": 'Algorithm 1 and Section 4',
        "note": 'Paper uses batch_size=100 per data_setup.',
        "paper_says": 'batch size B varying from  $\\{100,1000,10000\\}$ .',
    },
    "num_rounds": {
        "value": 5,
        "source": 'system_default',
        "paper_value": 349,
        "reasoning": (
            "Paper runs 349 rounds (~34,900 labels at batch_size=100). At smoke "
            "scale we run 5 rounds at runtime batch_size=100 (~500 labels) — enough "
            "to see a learning curve emerge while keeping the acquisition loop "
            "(which re-trains and re-scores the pool every round) cheap to run on a "
            "laptop CPU at smoke scale."
        ),
        "unused_by_method": True,
    },
    "initial_labeled": {
        "value": 100,
        "source": 'paper',
        "paper_section": 'Algorithm 1 and Section 4',
        "note": 'Paper bootstraps with 100 uniformly-random labeled examples.',
        "paper_says": 'M=100 being the number of initial random labeled examples',
        "unused_by_method": True,
    },
    "pool_size": {
        "value": 12000,
        "source": 'system_default',
        "paper_value": 'full training set',
        "reasoning": (
            "Paper uses the full training set (typically 60k–73k samples) as the "
            "unlabeled pool. We subsample to 12,000 to keep each round's "
            "acquisition-scoring pass over the whole pool cheap (the pool is "
            "re-scored every round, so this is the dominant cost of the acquisition "
            "loop). Raised further to 12,000 to satisfy the taxonomy "
            "smoke_economics max_budget_to_pool_ratio floor (0.05); a smaller pool "
            "would over-consume the unlabeled set and weaken diversity-sensitive "
            "acquisition checks. Note: this is a smoke-scale default, not a runtime "
            "estimate — the smoke gate is what actually verifies the notebook fits "
            "its per-cell time budget. Pool ratio (total_budget / pool_size) ≈ "
            "0.05. This is at or below the taxonomy smoke_economics "
            "`max_budget_to_pool_ratio` of 0.05 — diversity baselines are still "
            "meaningfully tested at this scale."
        ),
    },
    "learning_rate": {
        "value": 0.001,
        "source": 'paper',
        "paper_section": 'Section 4',
        "note": (
            "Adam optimizer at 0.001. Spec training.learning_rate: '0.001 for image "
            "data and 0.0001 for non-image data'. Selected image-data value."
        ),
    },
    "max_epochs": {
        "value": 50,
        "source": 'system_inferred',
        "reasoning": (
            "AUTO-RAISED (A001): max_epochs raised from 8 to 50. The "
            "training-budget sufficiency check found the derived budget could not "
            "fit a separable toy set sized to the round-one labeled set (train acc "
            "0.3333 vs chance+margin 0.5833); 50 epochs fit it with batch and "
            "learning rate unchanged. See assumptions.md entry A001. Previous "
            "reasoning: Paper trains to an explicit training-accuracy threshold "
            "with no stated epoch cap. At smoke scale the labeled sets are tiny (≤ "
            "~500 examples) and `train_until_accuracy` usually stops well before "
            "the cap; 8 is a low safety bound that keeps per-round training fast at "
            "smoke scale. Increase it for full-scale training."
        ),
    },
    "train_until_accuracy": {
        "value": 0.99,
        "source": 'paper',
        "paper_section": 'Section 4 EXPERIMENTS',
        "note": (
            "Paper-stated training threshold: 'models using cross-entropy loss and "
            "the Adam variant of SGD until training accuracy exceeds 99%'."
        ),
    },
    "hidden_dim": {
        "value": 256,
        "source": 'system_inferred',
        "reasoning": (
            "Paper states no data-type-keyed MLP width. We use hidden_dim=256 as "
            "the taxonomy-typical AL convention."
        ),
    },
}


def unpack(p: dict) -> dict:
    """Strip provenance and return a flat name -> value dict."""
    return {k: v["value"] for k, v in p.items() if v.get("used_in_notebook", True)}


cfg = unpack(params)

<a id="sec-optional-scale-up-to-paper-faithful-values"></a>

### Optional: scale up to paper-faithful values

The system-supplied round count, pool size, epoch cap, and MLP width are demo choices. The paper's image experiments use a 256-dimensional MLP embedding; full paper-scale reproduction also requires the paper datasets, repeated runs, and appropriate architecture conditions.

In [4]:
# cfg.update({
#     "num_rounds": 349,
#     "pool_size": 60000,
#     "max_epochs": 50,
#     "hidden_dim": 256,
# })
# True paper-faithfulness also requires matching the dataset and architecture condition.

<a id="sec-3-the-setup-pieces"></a>

## 3. The setup pieces

BADGE operates on ordinary supervised classification: labeled data, a multiclass model, and fresh training after each acquisition. These are paper protocol, **not** the paper's contribution; skim them and move to §4. The model must expose raw multiclass logits and `forward_with_embedding`, whose second output is the real penultimate activation.

<a id="sec-31-data"></a>

### 3.1 Data

`load_data` reads `method/example_data/` and downloads a flattened MNIST subset on first use. The runtime loader returns float32 features shaped `(N_pool, D)` and `(N_test, D)`, and int64 labels shaped `(N_pool,)` and `(N_test,)`; the architecture contract's smoke descriptors use `D=784` and `K=10`.

In [5]:
x_pool, y_pool, x_test, y_test = load_data(
    pool_size=cfg["pool_size"], n_test=1000, seed=SEED
)
input_dim = int(x_pool.shape[1])
n_classes = max(int(y_pool.max().item()), int(y_test.max().item())) + 1
print(f"pool: {tuple(x_pool.shape)} / {y_pool.dtype}")
print(f"test: {tuple(x_test.shape)} / {y_test.dtype}; classes: {n_classes}")

  0%|          | 0.00/9.91M [00:00<?, ?B/s]

  1%|          | 98.3k/9.91M [00:00<00:17, 572kB/s]

  2%|▏         | 197k/9.91M [00:00<00:13, 730kB/s] 

  4%|▍         | 426k/9.91M [00:00<00:07, 1.33MB/s]

  6%|▌         | 590k/9.91M [00:00<00:06, 1.35MB/s]

...

 91%|█████████▏| 1.51M/1.65M [00:00<00:00, 2.95MB/s]

100%|██████████| 1.65M/1.65M [00:00<00:00, 2.09MB/s]

  0%|          | 0.00/4.54k [00:00<?, ?B/s]

100%|██████████| 4.54k/4.54k [00:00<00:00, 5.32MB/s]

pool: (12000, 784) / torch.int64
test: (1000, 784) / torch.int64; classes: 10


<a id="sec-32-model"></a>

### 3.2 Model

The package supplies `GradientEmbeddingMLP`, a two-affine-layer MLP with hidden width `cfg["hidden_dim"]`. The paper evaluates MLP, ResNet-18, and VGG-11 conditions; this bundled MLP preserves the essential penultimate-layer hook and multiclass output, but does not reproduce those architecture results. Its contract forward input is float32 `(B, D)` and output is float32 `(B, K)`.

In [6]:
model = build_model(
    input_dim=input_dim, n_classes=n_classes, hidden_dim=cfg["hidden_dim"]
)
print(model)

GradientEmbeddingMLP(
  (encoder): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): ReLU()
  )
  (classifier): Linear(in_features=256, out_features=10, bias=True)
)


<a id="sec-33-training"></a>

### 3.3 Training

Each checkpoint uses Adam and cross-entropy, then retrains a **fresh** model after labels are acquired (**Algorithm 1, Section 4**). Training is performed in §3.4 and §5.1, so the checkpoint model always corresponds to the displayed labeled count.

<a id="sec-34-bootstrap-labeled-set"></a>

### 3.4 Bootstrap labeled set

BADGE starts from uniformly sampled labels, trains a warmup classifier, and uses that fixed snapshot for the component demonstrations below (**Algorithm 1, Section 3**). This is random initialization; BADGE has no separate method-specific core-set bootstrap.

<!-- derived-block: begin {"kind": "implementation_note", "spec": {"element_ids": ["eq-cross-entropy"], "file": "method/training.py", "qualname": "train_from_scratch"}} -->
**How this is implemented:** [`train_from_scratch`](method/training.py) (`method/training.py:22`) implements [Cross-entropy training loss](METHOD.md#eq-cross-entropy).

<details>
<summary>Source of <code>train_from_scratch</code> (37 lines)</summary>

**Source: `train_from_scratch` — method/training.py:22**

```python
def train_from_scratch(
    model: GradientEmbeddingMLP,
    x_train: torch.Tensor,
    y_train: torch.Tensor,
    *,
    learning_rate: float = 1e-3,
    max_epochs: int = 50,
    train_until_accuracy: float | None = 0.99,
    seed: int = 0,
) -> GradientEmbeddingMLP:
    """Train this freshly constructed classifier on the current labeled set."""
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_fn = nn.CrossEntropyLoss()

    # Full-batch gradient descent: the entire labeled set is one batch per epoch.
    # Fine for smoke-sized labeled sets (≤ ~1k examples). At paper-scale labeled
    # sets with the paper's full architecture, wrap (x_train, y_train) in a
    # DataLoader and step per mini-batch instead.
    model.train()
    for _ in range(max_epochs):
        optimizer.zero_grad()
        logits = model(x_train)
        loss = loss_fn(logits, y_train)
        # paper-element: eq-cross-entropy
        loss.backward()
        optimizer.step()
        if train_until_accuracy is not None:
            model.eval()
            with torch.no_grad():
                accuracy = (model(x_train).argmax(dim=1) == y_train).float().mean().item()
            model.train()
            if accuracy >= train_until_accuracy:
                break
    return model
```

</details>
<!-- derived-block: end -->

In [7]:
bootstrap_rng = np.random.default_rng(SEED)
bootstrap_labeled_idx = np.sort(
    bootstrap_rng.choice(len(x_pool), size=cfg["initial_labeled"], replace=False)
)
warmup_model = build_model(
    input_dim=input_dim, n_classes=n_classes, hidden_dim=cfg["hidden_dim"]
)
warmup_model = train_from_scratch(
    warmup_model,
    x_pool[bootstrap_labeled_idx],
    y_pool[bootstrap_labeled_idx],
    learning_rate=cfg["learning_rate"],
    max_epochs=cfg["max_epochs"],
    train_until_accuracy=cfg["train_until_accuracy"],
    seed=SEED,
)
warmup_model.eval()
with torch.no_grad():
    warmup_train_accuracy = (
        warmup_model(x_pool[bootstrap_labeled_idx]).argmax(dim=1)
        == y_pool[bootstrap_labeled_idx]
    ).float().mean().item()
    warmup_train_loss = F.cross_entropy(
        warmup_model(x_pool[bootstrap_labeled_idx]), y_pool[bootstrap_labeled_idx]
    ).item()
print(
    f"warmup training loss: {warmup_train_loss:.3f}; "
    f"accuracy: {warmup_train_accuracy:.3f}"
)

warmup training loss: 0.264; accuracy: 0.990


<a id="sec-4-the-badge-batch-active-learning-by-diverse-gradient-embeddings-method"></a>

## 4. The BADGE: Batch Active Learning by Diverse Gradient Embeddings method ⭐

This is the paper's contribution: predicted-label gradient embeddings followed by seeded k-MEANS++ sampling (**Algorithm 1, Section 3**). The demo preserves both core elements; it does not substitute scalar uncertainty scores or a different batch sampler.

<a id="sec-41-intuition"></a>

### 4.1 Intuition

Selecting only uncertain points can make a batch redundant, while selecting only diverse points can ignore likely model updates. BADGE uses the predicted label to form a last-layer gradient: its magnitude reflects uncertainty and its direction includes the penultimate representation. k-MEANS++ then favors separated high-impact embeddings without a hand-tuned trade-off (**Section 3**, **Algorithm 1**).

<a id="sec-42-hallucinated-gradient-embeddings"></a>

### 4.2 Hallucinated gradient embeddings ⭐

`compute_gradient_embeddings` uses the model prediction as the hypothetical label and returns one flattened final-layer gradient per candidate (**Algorithm 1, Section 3**). For class `i`, it computes $(p_i - \mathbb{1}\{\hat y=i\})z(x;V)$, so the implementation uses the actual penultimate activation rather than logits or input features (**Eq. 1, Section 3**).

<!-- derived-block: begin {"kind": "implementation_note", "spec": {"element_ids": ["eq-gradient-embedding", "eq-softmax-cross-entropy"], "file": "method/method.py", "qualname": "compute_gradient_embeddings"}} -->
**How this is implemented:** [`compute_gradient_embeddings`](method/method.py) (`method/method.py:20`) implements [BADGE gradient embedding](METHOD.md#eq-gradient-embedding), [Softmax cross-entropy at the final layer](METHOD.md#eq-softmax-cross-entropy).

<details>
<summary>Source of <code>compute_gradient_embeddings</code> (39 lines)</summary>

**Source: `compute_gradient_embeddings` — method/method.py:20**

```python
def compute_gradient_embeddings(model, x_unlabeled: Tensor) -> Tensor:
    """Construct BADGE hallucinated output-layer gradients (Eq. 1, Section 3).

    Args:
        model: Multiclass classifier exposing ``forward_with_embedding``.
        x_unlabeled: Candidate inputs in their original pool order.

    Returns:
        A ``(n_candidates, n_classes * embedding_dim)`` gradient tensor.
    """
    # paper-element: alg-gradient-embedding
    # paper-element: eq-gradient-embedding
    # essential: Penultimate-layer embedding hook
    # essential: Multiclass output layer
    was_training = model.training
    model.eval()
    try:
        with torch.no_grad():
            logits, embedding = model.forward_with_embedding(x_unlabeled)
            if logits.ndim != 2 or embedding.ndim != 2:
                raise ValueError(
                    "forward_with_embedding must return rank-2 logits and "
                    "penultimate embeddings"
                )
            if logits.shape[0] != embedding.shape[0]:
                raise ValueError("logits and embeddings must have the same batch size")

            # paper-element: eq-classifier-prediction
            predicted_labels = logits.argmax(dim=1)
            # paper-element: eq-softmax-cross-entropy
            probabilities = F.softmax(logits, dim=1)
            # paper-element: eq-softmax-gradient-block
            residuals = probabilities - F.one_hot(
                predicted_labels, num_classes=logits.shape[1]
            ).to(dtype=probabilities.dtype)
            # Each class block is (p_i - 1{y_hat=i}) z(x; V).
            return (residuals.unsqueeze(-1) * embedding.unsqueeze(1)).flatten(start_dim=1)
    finally:
        model.train(was_training)
```

</details>
<!-- derived-block: end -->

In [8]:
warmup_model.eval()
demo_candidates = x_pool[:cfg["batch_size"]]
gradient_embeddings = compute_gradient_embeddings(warmup_model, demo_candidates)
gradient_norms = gradient_embeddings.norm(dim=1)
print("gradient embedding shape:", tuple(gradient_embeddings.shape))
print("mean gradient norm:", float(gradient_norms.mean().item()))
plt.figure(figsize=(7, 3))
plt.hist(gradient_norms.detach().numpy(), bins=20)
plt.xlabel("gradient embedding norm")
plt.ylabel("candidate count")
plt.title("BADGE predicted-label gradient magnitudes")
plt.show()

gradient embedding shape: (100, 2560)
mean gradient norm: 4.095581531524658


<a id="sec-43-k-means-gradient-batch-seeding"></a>

### 4.3 k-MEANS++ gradient-batch seeding

`kmeans_plus_plus_seeding` first chooses one gradient embedding uniformly, then samples each later center in proportion to its squared distance from the nearest selected center (**Appendix A, Algorithm 2**). This incremental nearest-distance update makes selections diverse in BADGE's gradient-embedding space (**Section 3**, **Appendix A, Algorithm 2**).

<!-- derived-block: begin {"kind": "implementation_note", "spec": {"element_ids": ["alg-kmeanspp-seeding", "eq-kmeanspp-distance", "eq-kmeanspp-probability"], "file": "method/method.py", "qualname": "kmeans_plus_plus_seeding"}} -->
**How this is implemented:** [`kmeans_plus_plus_seeding`](method/method.py) (`method/method.py:61`) implements [k-MEANS++ seeding over gradient embeddings](METHOD.md#alg-kmeanspp-seeding), [Nearest-center distance](METHOD.md#eq-kmeanspp-distance), [k-MEANS++ sampling probability](METHOD.md#eq-kmeanspp-probability).

<details>
<summary>Source of <code>kmeans_plus_plus_seeding</code> (55 lines)</summary>

**Source: `kmeans_plus_plus_seeding` — method/method.py:61**

```python
def kmeans_plus_plus_seeding(
    embeddings: np.ndarray, batch_size: int, rng: np.random.Generator
) -> List[int]:
    """Sample BADGE batch positions by k-MEANS++ (Appendix A, Algorithm 2).

    Args:
        embeddings: One gradient embedding per candidate.
        batch_size: Requested number of unique positions.
        rng: Local seeded generator for the k-MEANS++ draws.

    Returns:
        Unique positions into ``embeddings``.
    """
    # paper-element: alg-kmeanspp-seeding
    if embeddings.ndim != 2:
        raise ValueError("embeddings must have shape (n_candidates, embedding_dim)")
    if batch_size < 0:
        raise ValueError("batch_size must be non-negative")

    n_candidates = embeddings.shape[0]
    n_select = min(batch_size, n_candidates)
    # paper-fidelity: Clamping only accommodates a smoke pool smaller than the
    # requested batch; at paper scale this is a no-op.
    if n_select == 0:
        return []

    selected = [int(rng.integers(n_candidates))]
    # paper-element: eq-kmeanspp-distance
    differences = embeddings - embeddings[selected[0]]
    nearest_squared_distance = np.einsum("ij,ij->i", differences, differences)

    while len(selected) < n_select:
        selected_mask = np.zeros(n_candidates, dtype=bool)
        selected_mask[selected] = True
        weights = nearest_squared_distance.copy()
        weights[selected_mask] = 0.0
        total_weight = float(weights.sum())

        # paper-element: eq-kmeanspp-probability
        if total_weight > 0.0 and np.isfinite(total_weight):
            next_index = int(rng.choice(n_candidates, p=weights / total_weight))
        else:
            # Equal embeddings make Algorithm 2's all-zero distribution
            # undefined; sample uniformly from the still-unselected indices.
            remaining = np.flatnonzero(~selected_mask)
            next_index = int(remaining[rng.integers(len(remaining))])

        selected.append(next_index)
        differences = embeddings - embeddings[next_index]
        new_squared_distance = np.einsum("ij,ij->i", differences, differences)
        nearest_squared_distance = np.minimum(
            nearest_squared_distance, new_squared_distance
        )

    return selected
```

</details>
<!-- derived-block: end -->

In [9]:
demo_embedding_array = gradient_embeddings.detach().numpy()
demo_rng = np.random.default_rng(SEED)
demo_selected_positions = kmeans_plus_plus_seeding(
    demo_embedding_array, cfg["batch_size"], demo_rng
)
print(f"selected {len(demo_selected_positions)} unique positions")
print("first ten positions:", demo_selected_positions[:10])

selected 100 unique positions
first ten positions: [8, 46, 86, 75, 9, 94, 77, 79, 11, 48]


<a id="sec-44-putting-it-together"></a>

### 4.4 Putting it together

`select_batch(model, x_unlabeled, batch_size, seed)` composes the two helpers: it computes predicted-label gradient embeddings for the exact candidate tensor, then performs seeded k-MEANS++ and returns positions into that same tensor (**Algorithm 1**, **Section 3**).

```python
positions = select_batch(model, x_unlabeled, batch_size, seed)
```

<a id="sec-5-running-active-learning-end-to-end"></a>

## 5. Running active learning end-to-end

The tutorial continues from the bootstrap state. It evaluates the initial model once, then performs one BADGE acquisition, label merge, fresh retraining, and evaluation for each acquisition round (**Algorithm 1, Section 3**). Each printed round metric is test accuracy, the fraction of correct predicted test labels.

<a id="sec-51-the-acquisition-loop"></a>

### 5.1 The acquisition loop

The loop maps pool-relative positions returned by `select_batch` back through `unlabeled_idx`. It rebuilds the model after every merge, so no checkpoint warm-starts a later round (**Algorithm 1, Section 4**).

<!-- derived-block: begin {"kind": "implementation_note", "spec": {"element_ids": ["alg-badge", "concept-diverse-uncertain-batches", "concept-pool-based-active-learning"], "file": "method/method.py", "qualname": "select_batch"}} -->
**How this is implemented:** [`select_batch`](method/method.py) (`method/method.py:118`) implements [BADGE batch active learning](METHOD.md#alg-badge), [Joint diversity and uncertainty acquisition](METHOD.md#concept-diverse-uncertain-batches), [Pool-based active learning setting](METHOD.md#concept-pool-based-active-learning).

<details>
<summary>Source of <code>select_batch</code> (22 lines)</summary>

**Source: `select_batch` — method/method.py:118**

```python
def select_batch(model, x_unlabeled, batch_size, seed) -> List[int]:
    """Select a diverse uncertain BADGE batch (Algorithm 1, Section 3).

    Args:
        model: Trained multiclass classifier for this acquisition round.
        x_unlabeled: Candidate pool whose positions are returned.
        batch_size: Requested number of candidates.
        seed: Per-round seed for local k-MEANS++ randomness.

    Returns:
        Unique candidate positions in the input pool.
    """
    # paper-element: alg-badge
    # paper-element: concept-diverse-uncertain-batches
    # paper-element: concept-pool-based-active-learning
    if x_unlabeled.shape[0] == 0 or batch_size <= 0:
        return []

    gradient_embeddings = compute_gradient_embeddings(model, x_unlabeled)
    embedding_array = gradient_embeddings.detach().cpu().numpy()
    rng = np.random.default_rng(seed)
    return kmeans_plus_plus_seeding(embedding_array, batch_size, rng)
```

</details>
<!-- derived-block: end -->

In [10]:
def evaluate(model, x, y):
    model.eval()
    with torch.no_grad():
        return (model(x).argmax(dim=1) == y).float().mean().item()


def training_loss(model, x, y):
    model.eval()
    with torch.no_grad():
        return F.cross_entropy(model(x), y).item()


labeled_idx = bootstrap_labeled_idx.copy()
unlabeled_idx = np.setdiff1d(np.arange(len(x_pool)), labeled_idx)
learning_curve = []

initial_model = build_model(
    input_dim=input_dim, n_classes=n_classes, hidden_dim=cfg["hidden_dim"]
)
initial_model = train_from_scratch(
    initial_model,
    x_pool[labeled_idx],
    y_pool[labeled_idx],
    learning_rate=cfg["learning_rate"],
    max_epochs=cfg["max_epochs"],
    train_until_accuracy=cfg["train_until_accuracy"],
    seed=SEED,
)
initial_accuracy = evaluate(initial_model, x_test, y_test)
initial_loss = training_loss(initial_model, x_pool[labeled_idx], y_pool[labeled_idx])
learning_curve.append((len(labeled_idx), initial_accuracy))
print(
    f"round 0: labels={len(labeled_idx)}, training loss={initial_loss:.3f}, "
    f"test accuracy={initial_accuracy:.3f}"
)

current_model = initial_model
for round_index in range(cfg["num_rounds"]):
    current_model.eval()
    positions = select_batch(
        model=current_model,
        x_unlabeled=x_pool[unlabeled_idx],
        batch_size=cfg["batch_size"],
        seed=SEED + round_index,
    )
    chosen = unlabeled_idx[np.asarray(positions, dtype=int)]
    labeled_idx = np.union1d(labeled_idx, chosen)
    unlabeled_idx = np.setdiff1d(unlabeled_idx, chosen)

    current_model = build_model(
        input_dim=input_dim, n_classes=n_classes, hidden_dim=cfg["hidden_dim"]
    )
    current_model = train_from_scratch(
        current_model,
        x_pool[labeled_idx],
        y_pool[labeled_idx],
        learning_rate=cfg["learning_rate"],
        max_epochs=cfg["max_epochs"],
        train_until_accuracy=cfg["train_until_accuracy"],
        seed=SEED + round_index + 1,
    )
    round_accuracy = evaluate(current_model, x_test, y_test)
    round_loss = training_loss(current_model, x_pool[labeled_idx], y_pool[labeled_idx])
    learning_curve.append((len(labeled_idx), round_accuracy))
    print(
        f"round {round_index + 1}: labels={len(labeled_idx)}, "
        f"selected={len(chosen)}, training loss={round_loss:.3f}, "
        f"test accuracy={round_accuracy:.3f}"
    )

round 0: labels=100, training loss=0.264, test accuracy=0.731


round 1: labels=200, selected=100, training loss=0.179, test accuracy=0.821


round 2: labels=300, selected=100, training loss=0.197, test accuracy=0.843


round 3: labels=400, selected=100, training loss=0.169, test accuracy=0.852


round 4: labels=500, selected=100, training loss=0.214, test accuracy=0.873


round 5: labels=600, selected=100, training loss=0.257, test accuracy=0.876


<a id="sec-52-learning-curve"></a>

### 5.2 Learning curve

This is one BADGE run on one smoke-sized MNIST condition, not an estimate of the paper's multi-run, multi-condition results (**Section 4**).

In [11]:
labels_acquired, test_accuracies = zip(*learning_curve)
plt.figure(figsize=(7, 4))
plt.plot(labels_acquired, test_accuracies, marker="o", label="BADGE")
plt.xlabel("total labeled examples")
plt.ylabel("test accuracy")
plt.title("BADGE on smoke-sized MNIST (single seed)")
plt.ylim(0.0, 1.0)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

<a id="sec-6-use-your-own-data"></a>

## 6. Use your own data

**Option A:** pass a `.pt` file or directory to `load_data(path=...)`. A `.pt` file stores `x_pool`, `y_pool`, `x_test`, and `y_test`; features are float tensors and labels are long tensors.

**Option B:** place data in `method/example_data/`. The loader accepts the same four-key `.json` structure, or paired `pool.csv` (or `train.csv`) and `test.csv` files with float features followed by an integer label in the final column.

Inputs must be flattened 2D `(N, input_dim)` tensors for the bundled MLP. For image-shaped inputs, replace the architecture with a compatible CNN that returns raw multiclass logits and the real penultimate activation through `forward_with_embedding`.